# 04 - Clusterizacao de Voos

Agrupamento para entender perfis operacionais sem usar o alvo.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

plt.style.use("seaborn-v0_8")
data = pd.read_parquet("../data/processed/flights_sample.parquet")
features = ["MONTH","DAY_OF_WEEK","DEP_HOUR","DISTANCE"]
X = data[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X.shape

## 1. M?todo do cotovelo (leve)

In [ ]:
inertia = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)
plt.plot(range(2,7), inertia, marker="o")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.title("Cotovelo (k de 2 a 6)")
plt.show()

## 2. Ajuste final (k=3)

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)
data["CLUSTER"] = clusters

perfil = data.groupby("CLUSTER")[features + ["DELAYED"]].agg({
    "MONTH":"mean",
    "DAY_OF_WEEK":"mean",
    "DEP_HOUR":"mean",
    "DISTANCE":"mean",
    "DELAYED":"mean"
})
print(perfil)

## 3. Taxa de atraso por cluster

In [ ]:
sns.barplot(data=data, x="CLUSTER", y="DELAYED")
plt.title("Atraso (>15min) por cluster")
plt.show()

### Leituras
- Clusters distinguem hor?rios m?dios e dist?ncias.
- Taxas de atraso variam entre grupos, sugerindo perfis operacionais com maior risco mesmo sem usar a vari?vel alvo.